In [0]:
%python
# Strip whitespace from column names
df = spark.sql("select * from new_sample.retail_data.customer_sales_silver")
df = df.toDF(*[col.strip() for col in df.columns])

# Standardize customer names to title case and strip whitespace
from pyspark.sql.functions import trim, initcap, col

df = df.withColumn("customer_name", initcap(trim(col("customer_name"))))

# Convert data types for IDs, numeric columns, and dates
from pyspark.sql.functions import *

df = df.withColumn("customer_id", col("customer_id").cast("long")) \
       .withColumn("units_purchased", col("units_purchased").cast("double")) \
       .withColumn("total_price", col("total_price").cast("double")) \
       .withColumn("order_date", to_date(col("order_date")))

# Drop rows with missing values after conversions
df = df.dropna(subset=["customer_id", "units_purchased", "total_price", "order_date"])

display(df)


In [0]:
%python
# --- Feature Engineering ---

from pyspark.sql.functions import year, month, date_format, round as spark_round, col

# Extract order year and order month for time-series analyses
df = df.withColumn("order_year", year(col("order_date"))) \
       .withColumn("order_month", date_format(col("order_date"), "MM"))

# Calculate average price per unit and round to 2 decimals
df = df.withColumn("avg_price_per_unit", spark_round(col("total_price") / col("units_purchased"), 2))

display(df)

In [0]:
%python
df.write.mode("overwrite").saveAsTable("new_sample.retail_data.customer_sales_gold")